# Premiers pas avec les *embeddings* de phrases

Un modèle d'*embeddings* transforme un texte en un vecteur, de sorte que deux textes de sens proche aient des vecteurs proches. Le modèle de ces travaux pratiques : [Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B), multilingue, 1024 dimensions, l'un des meilleurs modèles ouverts de sa taille.

1. Comparer une douzaine de phrases
2. Chercher dans un mois de presse, avec et sans instruction
3. Chercher dans une autre langue
4. Cosinus, produit scalaire, distance euclidienne
5. Tronquer les vecteurs (Matryoshka)

Ce notebook a besoin d'un GPU : *Exécution > Modifier le type d'exécution > GPU T4*.

## Données et modèle

Les passages de la presse de 1914 (voir le TP précédent), et le modèle, chargé en demi-précision (`half()`) : deux fois moins de mémoire, plus rapide sur GPU.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/passages.parquet

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device="cuda").half()
# Le modèle accepte 32 000 tokens : on coupe à 512, assez pour nos passages, sinon
# les plus longs saturent la mémoire du GPU.
model.max_seq_length = 512

## 1. Comparer une douzaine de phrases

Encodez les phrases de `SENTENCES` avec `model.encode_document`, calculez la matrice de leurs similarités avec `model.similarity`, et affichez-la avec `plt.imshow` (les phrases en étiquettes des axes). Quelles paires obtiennent les meilleurs scores ? Trouvez deux surprises.

In [ ]:
SENTENCES = [
    "Le VPN se déconnecte sans arrêt",
    "Impossible de me connecter au réseau depuis chez moi",
    "My VPN keeps disconnecting",
    "J'ai reçu un e-mail suspect",
    "Un message frauduleux est arrivé dans ma boîte",
    "L'imprimante n'imprime plus",
    "Mon écran reste noir",
    "Comment poser des congés ?",
    "Quand arrive la fiche de paie ?",
    "Le serveur fonctionne",
    "Le serveur ne fonctionne pas",
    "La cantine ferme à 14 h",
]

In [ ]:
# Votre code ici

### Solution

In [ ]:
vectors = model.encode_document(SENTENCES)
similarities = model.similarity(vectors, vectors).numpy()

fig, ax = plt.subplots(figsize=(9, 8))
image = ax.imshow(similarities, cmap="viridis")
ax.set_xticks(range(len(SENTENCES)), SENTENCES, rotation=90)
ax.set_yticks(range(len(SENTENCES)), SENTENCES)
fig.colorbar(image)
plt.show()

# - Une phrase et sa traduction anglaise sont presque confondues : le modèle est multilingue.
# - « Le serveur fonctionne » et « Le serveur ne fonctionne pas » sont très proches : les
#   embeddings captent le sujet bien mieux qu'une négation.
# - Deux phrases sans rapport ne tombent pas à 0 : seule la comparaison des scores a un sens.

## 2. Chercher dans un mois de presse

Gardez les passages publiés entre le 20 juillet et le 10 août 1914 (`date` est une date), puis encodez leurs textes avec `model.encode_document` (`batch_size=32`, `show_progress_bar=True`).

Écrivez ensuite une fonction `search(query, k=5, instruction=True)` qui affiche la date, le score et le début des `k` passages les plus proches de la requête. Avec `instruction=True`, encodez la requête avec `model.encode_query`, qui la précède d'une instruction (« *Given a web search query, retrieve relevant passages that answer the query* ») ; sinon avec `model.encode`, sans instruction.

In [ ]:
# Votre code ici

### Solution

In [ ]:
passages = pd.read_parquet("passages.parquet")
month = passages[passages.date.between("1914-07-20", "1914-08-10")].reset_index(drop=True)
print(len(month), "passages")
month_vectors = model.encode_document(month.text.tolist(), batch_size=32, show_progress_bar=True)


def search(query: str, k: int = 5, instruction: bool = True) -> None:
    query_vector = model.encode_query(query) if instruction else model.encode(query)
    scores = model.similarity(query_vector, month_vectors)[0].numpy()
    for i in np.argsort(-scores)[:k]:
        print(f"{month.date[i]:%d/%m} {scores[i]:.3f} {month.text[i][:150]}")

Cherchez, avec et sans instruction :

- l'assassinat de Jean Jaurès
- la mobilisation générale
- le verdict du procès de Mme Caillaux

Les résultats sont-ils pertinents ? L'instruction change-t-elle quelque chose ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
for query in ("l'assassinat de Jean Jaurès", "la mobilisation générale", "le verdict du procès de Mme Caillaux"):
    for instruction in (True, False):
        print(f"--- {query} ({'avec' if instruction else 'sans'} instruction)")
        search(query, instruction=instruction)

# Avec l'instruction, la requête est encodée comme une question qui cherche sa réponse :
# les résultats sont en général meilleurs, et ses scores plus élevés. Sans elle, une requête
# courte cherche des textes qui lui ressemblent, pas des textes qui y répondent.

## 3. Chercher dans une autre langue

Le corpus est en français : cherchez-y avec des requêtes en anglais, par exemple « *the assassination of Jean Jaurès* » ou « *Germany declares war on France* ». Et pourquoi pas dans une autre langue que vous connaissez ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
search("the assassination of Jean Jaurès")
search("Germany declares war on France")
search("Deutschland erklärt Frankreich den Krieg")

## 4. Cosinus, produit scalaire, distance euclidienne

Calculez la norme des vecteurs de `month_vectors` (`np.linalg.norm`). Puis, pour la requête « la mobilisation générale », classez les passages selon le cosinus, le produit scalaire et la distance euclidienne (du plus proche au plus loin) : les trois classements ont-ils les mêmes 10 premiers ? Pourquoi ? Quand ne serait-ce plus le cas ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
vectors = month_vectors.astype(np.float32)
print(np.linalg.norm(vectors, axis=1)[:5])

query_vector = model.encode_query("la mobilisation générale").astype(np.float32)
dot = vectors @ query_vector
cosine = dot / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(query_vector))
euclidean = np.linalg.norm(vectors - query_vector, axis=1)
rankings = {
    "cosinus": np.argsort(-cosine)[:10],
    "produit scalaire": np.argsort(-dot)[:10],
    "euclidienne": np.argsort(euclidean)[:10],
}
for name, ranking in rankings.items():
    print(f"{name:17}", ranking)

# Le modèle normalise ses vecteurs (norme 1) : le produit scalaire est alors le cosinus, et
# |u - v|² = 2 - 2 cos(u, v), donc la distance euclidienne classe dans le même ordre. Avec des
# vecteurs de normes différentes, le produit scalaire favoriserait les plus longs.

## 5. Tronquer les vecteurs (Matryoshka)

Le modèle est entraîné pour que les **premières** dimensions portent l'essentiel du sens. Tronquez les vecteurs des passages et des requêtes à 512, 256, 128 et 64 dimensions (puis renormalisez-les), et mesurez, pour 200 passages pris au hasard comme requêtes, la part de leurs 10 plus proches voisins (en 1024 dimensions) qu'on retrouve après troncature. Combien de mémoire faudrait-il pour 1 million de passages à chaque taille, en `float32` ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
def truncate(x: np.ndarray, dims: int) -> np.ndarray:
    x = x[:, :dims]
    return x / np.linalg.norm(x, axis=1, keepdims=True)


rng = np.random.default_rng(0)
queries = rng.choice(len(vectors), 200, replace=False)


def neighbours(x: np.ndarray) -> np.ndarray:
    scores = x[queries] @ x.T
    scores[np.arange(len(queries)), queries] = -np.inf  # un passage n'est pas son propre voisin
    return np.argsort(-scores, axis=1)[:, :10]


reference = neighbours(vectors)
for dims in (512, 256, 128, 64):
    found = neighbours(truncate(vectors, dims))
    overlap = np.mean([len(set(a) & set(b)) / 10 for a, b in zip(reference, found)])
    print(f"{dims:4} dimensions : {overlap:.0%} des voisins retrouvés, {dims * 4 * 1e6 / 1e9:.1f} Go par million de passages")